# **Welcome to the Flux tutorial at ISC High Performance 2025!**
<h1><center><img src="https://wihobbs.github.io/flux%20at%20isc.png" style="max-width: 1000px;" alt="Flux at ISC 2025 merged logos" /></center></h1>

Thanks for joining us today! We're stoked to be sharing an overview, resources, tips and tricks about Flux with you.

### **Tutorial overview**

* **Chapter 1 (You are here!⭐)**: Getting Started with Flux, Job Submission and Workflow Examples
  * Flux documentation overview and pointers
  * `flux resource` and resource discovery
  * `flux start`, `flux run`, `flux submit`, `flux batch`, `flux alloc`
  * Tuning and optional arguments for submission commands
* **Chapter 2**: 
  * `python3 -c "import flux"` (Flux's python bindings)
  * Synchronous job submission and monitoring
  * the FluxExecutor
* **Chapter 3**: Working within a Flux instance, manipulating and monitoring jobs
  * Monitoring job status/state: `flux job info`, `flux job`, `flux job last`, customizable output
  * JournalConsumer interface from python
  * Flux's config file (applicable to batch and alloc instances)

#### **A word on Jupyter Notebooks**

To go through this tutorial, you need to go through these modules in the order (1-3). To step through examples in each module's notebook, you need to execute cells. To run a cell, press `Shift+Enter` on your keyboard. If you prefer, you can also paste the shell commands in the JupyterLab terminal and execute them there.

Each command in a Jupyter cell is prepended with `!`, that's not necessary if working directly out of the terminal. Python code in a Jupyter cell is portable to another Python interpreter or `.py` file.

#### **Getting Started with Flux**

As you progress through the tutorial, our **documentation** can be hugely helpful in finding the exact right command/flag/option to use. Here's where we usually start when working with the Flux documentation:

1. The [flux-core Manual Pages](https://flux-framework.readthedocs.io/projects/flux-core/en/latest/index_man.html) will contain in-depth descriptions of most Flux commands. They are also available using the `man(1)` command.
2. If coming from previous experience with other resource managers, the [Batch System Cross-Reference Guide](https://hpc.llnl.gov/banks-jobs/running-jobs/batch-system-cross-reference-guides) can be helpful.
3. You can type `flux help` or even `flux help [COMMAND]`.


In [9]:
!flux help

Usage: flux [OPTIONS] COMMAND ARGS
  -h, --help             Display this message.
  -v, --verbose          Be verbose about environment and command search
  -V, --version          Display command and component versions
  -p, --parent           Set environment of parent instead of current instance
  -r, --root             Set environment of root instead of current instance

For general Flux documentation, please visit
    https://flux-framework.readthedocs.io

run and submit jobs, allocate resources
   submit             submit a job to a Flux instance
   run                run a Flux job interactively
   bulksubmit         submit jobs in bulk to a Flux instance
   alloc              allocate a new Flux instance for interactive use
   batch              submit a batch script to Flux

list and interact with jobs
   jobs               list jobs submitted to Flux
   top                display running Flux jobs
   pstree             display job hierarchies
   cancel             cancel one o

In [ ]:
!flux help submit

#### **You get an Instance!**

Even the most basic levels of `flux` commands will require a Flux instance to be running to execute them. A **flux instance** is a distributed application of _brokers_ sending _messages_ up and down a network. When you `start` a flux instance, you get at least one broker, and broker _modules_ like the scheduler, key-value store, and job manager, all of which help you run your own mini cluster inside a flux instance!

We have started one for you, but with a `flux` binary you could start one locally by typing `flux start`. We developers often start instances with the `-s` flag, which will give you a test instance with a fake number of nodes. 

In [4]:
!flux start --test-size=4 flux resource list

     STATE NNODES NCORES NGPUS NODELIST
      free      4     64     0 auk108.llnl.gov,auk108.llnl.gov,auk108.llnl.gov,auk108.llnl.gov
 allocated      0      0     0 
      down      0      0     0 


For interacting with this notebook, we've started a Flux instance for you already. Within your instance, you control _everything_! You can submit jobs. You can allow other people to connect to you and submit jobs. You can stop the queues (in your instance). You can start the queues (in your instance). You can submit 100,000 jobs on a 1-core instance and the silliness of doing that _only affects your instance._ We'll show you how to do these things through this tutorial.

In [10]:
## Let's start another instance, and submit a bunch of jobs to it
!flux start flux submit --cc=1-10 hostname
!echo "Instance is over"
!flux jobs -a -u hobbs17

****************************************************************************
* hwloc 2.11.2 received invalid information from the operating system.
*
* Failed with error: intersection without inclusion
* while inserting Group0 (groupkind 1000-0 cpuset 0x00ff0000,0x0)
* at Package (P#0 cpuset 0xffffffff,0xffffffff nodeset 0x0000000f)
* coming from: topology:io_parent
*
* The following FAQ entry in the hwloc documentation may help:
*   What should I do when hwloc reports "operating system" warnings?
* Otherwise please report this error message to the hwloc user's mailing list,
* along with the files generated by the hwloc-gather-topology script.
* 
* hwloc will now ignore this invalid topology information and continue.
****************************************************************************
febpnYF
febpnYG
febpnYH
fedJmpb
fedJmpc
fedJmpd
fedJmpe
feenm6w
feenm6x
feenm6y
Instance is over
       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO


We submitted 10 jobs to a flux instance above, but none of them showed up in the instance on which this notebook is running, because they were contained to a _different_ instance.

> **Caution**

> Don't type `!flux start` _without arguments_ in a Jupyter Notebook. The default behavior of `!flux start` is to open a shell that blocks, and this doesn't play nice with Jupyter's cells. Typing something like `!flux start hostname` or `!flux start flux resource list` is totally fine.

#### **Resource and Queue Status**

It's helpful to start visualizing our Flux instance by looking at the resources it has available.

In [12]:
!flux resource list

     STATE QUEUE       PROPERTIES NNODES NCORES NGPUS NODELIST
      free pci,pall                    4    256    32 tioga[12-15]
      free pdebug,pall                15    960   120 tioga[26-40]
 allocated pdebug,pall                 8    512    64 tioga[18-25]
      down pci,pall                    2    128    16 tioga[16-17]
      down pdebug,pall                 1     64     8 tioga41
      down pall        mi300a          2    192     8 tioga[42-43]


In [13]:
!flux start flux resource list

****************************************************************************
* hwloc 2.11.2 received invalid information from the operating system.
*
* Failed with error: intersection without inclusion
* while inserting Group0 (groupkind 1000-0 cpuset 0x00ff0000,0x0)
* at Package (P#0 cpuset 0xffffffff,0xffffffff nodeset 0x0000000f)
* coming from: topology:io_parent
*
* The following FAQ entry in the hwloc documentation may help:
*   What should I do when hwloc reports "operating system" warnings?
* Otherwise please report this error message to the hwloc user's mailing list,
* along with the files generated by the hwloc-gather-topology script.
* 
* hwloc will now ignore this invalid topology information and continue.
****************************************************************************
     STATE NNODES NCORES NGPUS NODELIST
      free      1     64     8 tioga10
 allocated      0      0     0 
      down      0      0     0 


In [14]:
!flux resource status

       STATE UP NNODES NODELIST
       avail  ✔     27 tioga[12-15,18-40]
     exclude  ✔      3 tioga[6,10-11]
    drained*  ✗      4 tioga[16-17,42-43]
     drained  ✔      1 tioga41


It might also be useful to look at queues. Every Flux instance starts with one anoymous queue for all resources.

In [15]:
!flux queue status && flux queue list

pdebug: Job submission is enabled
pdebug: Scheduling is started
mi300a: Job submission is disabled: nodes retired
mi300a: Scheduling is stopped
pci: Job submission is enabled
pci: Scheduling is started
pall: Job submission is enabled
pall: Scheduling is stopped
QUEUE    EN ST TDEFAULT   TLIMIT     NNODES     NCORES      NGPUS
pdebug*   ✔  ✔      30m      12h       0-24     0-1536      0-192
pci       ✔  ✔      30m       2h        0-6      0-384       0-48
pall      ✔  ✗      30m       1d       0-32     0-2112      0-248


#### **All the Different Ways to Do Work (from the CLI)**
Here's a basic table that shows the four submission commands we use in Flux. 

|                        | creates subinstance           | runs distributed application          |
|------------------------|-------------------------------|---------------------------------------|
| interactive            | `flux alloc`                  | `flux run`                            |
| backgrounded           | `flux batch`                  | `flux submit`👀                       |

* `flux alloc` will allocate resources and start an interactive Flux sub-instance underneath those resources. Within that subinstance, you can submit as many jobs as you like, with no worry about backing up the parent (usually system) instance.
* `flux batch` will also allocate resources and start a Flux sub-instance, but the job is not interactive, and thus `batch` requires a script outlining the work to do.
* `flux run` runs a program under a Flux instance. It does not create a new sub-instance, and will watch until the program completes.
* `flux submit` does not exist in other resource managers, notably Slurm. It does the same thing as `flux run`, but does not watch for job output, instead writing this to a file. 

It can be kind of hard to think about this in the abstract, so here's a program that we'll run under each of the commands to display different behavior.

```c
#include <mpi.h>
#include <stdio.h>
#include <unistd.h>
#include <stdlib.h>
#include <time.h>
#include <stdbool.h>

static struct timespec ts_diff (struct timespec start, struct timespec end)
{
        struct timespec temp;
        if ((end.tv_nsec-start.tv_nsec)<0) {
                temp.tv_sec = end.tv_sec-start.tv_sec-1;
                temp.tv_nsec = 1000000000+end.tv_nsec-start.tv_nsec;
        } else {
                temp.tv_sec = end.tv_sec-start.tv_sec;
                temp.tv_nsec = end.tv_nsec-start.tv_nsec;
        }
        return temp;
}

double monotime_since (struct timespec t0)
{
    struct timespec ts, d;
    clock_gettime (CLOCK_MONOTONIC, &ts);

    d = ts_diff (t0, ts);

    return ((double) d.tv_sec * 1000 + (double) d.tv_nsec / 1000000);
}

void monotime (struct timespec *tp)
{
    clock_gettime (CLOCK_MONOTONIC, tp);
}

bool monotime_isset (struct timespec t)
{
    return (t.tv_sec || t.tv_nsec);
}

int main (int argc, char *argv[])
{
    int id, ntasks;
    struct timespec t;
    const char *label;

    if (!(label = getenv ("FLUX_JOB_CC")))
        if (!(label = getenv ("FLUX_JOB_ID")))
            label = "0";

    monotime (&t);
    MPI_Init (&argc, &argv);
    MPI_Comm_rank (MPI_COMM_WORLD, &id);
    MPI_Comm_size (MPI_COMM_WORLD, &ntasks);
    if (id == 0) {
        printf ("%s: completed MPI_Init in %0.3fs.  There are %d tasks\n",
                label,
                monotime_since (t) / 1000, ntasks);
        fflush (stdout);
    }

    monotime (&t);
    MPI_Barrier (MPI_COMM_WORLD);
    if (id == 0) {
        printf ("%s: completed first barrier in %0.3fs\n",
                label,
                monotime_since (t) / 1000);
        fflush (stdout);
    }

    monotime (&t);
    MPI_Finalize ();
    if (id == 0) {
        printf ("%s: completed MPI_Finalize in %0.3fs\n",
                label,
                monotime_since (t) / 1000);
        fflush (stdout);
    }
    return 0;
}
```

Try not to dwell too much on what the program is doing -- it's basically an MPI "Hello, World" with a barrier in the middle. Let's try it with `flux run` first.


In [6]:
!flux run -N1 -n16 ./hellompi

ƒG9gyLQo1: completed MPI_Init in 0.726s.  There are 16 tasks
ƒG9gyLQo1: completed first barrier in 0.000s
ƒG9gyLQo1: completed MPI_Finalize in 1.012s


The `-N 1` flag told Flux to allocate one node, and the `-n 16` flag told Flux to run 16 tasks, hence there were 16 MPI tasks communicating in `MPI_COMM_WORLD`. Let's try the same flags with `flux alloc`.

In [8]:
!flux alloc -N1 -n16 flux run -n16 ./hellompi

ƒAL8Urs: completed MPI_Init in 0.732s.  There are 16 tasks
ƒAL8Urs: completed first barrier in 0.001s
ƒAL8Urs: completed MPI_Finalize in 1.012s
[detached: session exiting]


The `-N1 -n16` flags mean something different in this context. The `./hellompi` is passed as the "initial program" to `flux alloc`, which allocates 1 node with 16 cores, then starts the initial program -- only one copy of it. Think about it, this makes sense for `flux batch` which must take a script: 

In [9]:
!flux batch -N1 -n16 ./hellompi

flux-batch: ERROR: ./hellompi does not appear to be a script, or failed to encode as utf-8


In [10]:
!flux batch -N1 -n16 --wrap ./hellompi

ƒGp1Y7jE7


In [11]:
!flux watch $(flux job last)

0: stdout redirected to flux-ƒGp1Y7jE7.out
0: stderr redirected to flux-ƒGp1Y7jE7.out


In [12]:
## Use this to `cat` the file
!cat flux-ƒGp1Y7jE7.out

0: completed MPI_Init in 0.532s.  There are 1 tasks
0: completed first barrier in 0.000s
0: completed MPI_Finalize in 1.008s


#### What's this `submit` thing again?

We're glad you asked, because it's something that doesn't exist in other resource managers. `submit` is roughly equivalent to a backgrounded `flux run`.

In [18]:
!flux submit -n16 ./hellompi

ƒdqfmNtA3


In [19]:
!flux watch $(flux job last)

ƒdqfmNtA3: completed MPI_Init in 0.727s.  There are 16 tasks
ƒdqfmNtA3: completed first barrier in 0.008s
ƒdqfmNtA3: completed MPI_Finalize in 1.010s


#### Some submission flags of note

* `-N` specifies a number of nodes
* `-n` specifies a number of tasks for distributed applications, and cores for interactive allocations
* `-c` specifies a number of cores per task
* `--requires` constrains a job to run on a specific rank or hostname
* `--dependency` makes a job depend on another job
* `-cc` submits carbon copies of the same job many times
* `--out` and `--err` redirect output and error to files

Let's work through some examples of these flags!

In [17]:
!flux submit -N1 -n4 -c4 --requires=hosts:auk108.llnl.gov --out /tmp/file.txt --err /dev/null ./hellompi

ƒM46cVPeP


In [18]:
!flux run -N1 --dependency=afterok:$(flux job last) cat /tmp/file.txt

ƒM46cVPeP: completed MPI_Init in 0.485s.  There are 4 tasks
ƒM46cVPeP: completed first barrier in 0.000s
ƒM46cVPeP: completed MPI_Finalize in 1.008s


#### I need more jobs! This one-job-per-command example is too slow!

We can use the `--cc` flag to submit carbon copies of jobs!

In [20]:
!flux submit --cc=1-4 hostname

ƒMevUguM1
ƒMevUguM2
ƒMevUguM3
ƒMevUguM4


In [21]:
!seq 1 4 | flux bulksubmit --out=/tmp/{}.txt hostname

ƒMgez4DQs
ƒMgez4DQt
ƒMgez4DQu
ƒMgez4DQv


In [51]:
!flux watch $(flux job last)

0: stdout redirected to /tmp/100.txt
0: stderr redirected to /tmp/100.txt


In [22]:
!cat /tmp/1.txt

auk108.llnl.gov


#### **We just did a lot of work!** What does it look like?

We can use `flux jobs` to inspect the log of our running and finished jobs.

In [23]:
!flux jobs

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO


It's empty by default because we don't have any jobs currently running! But if we want to see our old jobs `-a`.

In [1]:
!flux jobs -a

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO


**But I want to see a running job!**

Ok, this is where our handy `submit` command is helpful!

In [25]:
!flux submit -n1 sleep inf

ƒQBrjcStw


In [26]:
!flux jobs

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
   ƒQBrjcStw hobbs17  sleep       R      1      1   3.517s auk108.llnl.gov


**Cool! Now kill that job! 💀**

We can use either `flux cancel $(flux job last)`, `flux cancel [JOBID]`, or `flux cancel --all`.

In [27]:
!flux cancel --all

flux-cancel: Canceled 1 job (0 errors)


In [8]:
!flux jobs

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
   ƒ6cyD4Dyh hobbs17  sleep       R      1      1   2.495s auk108.llnl.gov


In [9]:
!flux cancel --all

flux-cancel: Canceled 1 job (0 errors)


#### **Even More to Configure!** (the Job Shell)

A lot of the tuning batch end-users might like to have is in [Job Shell Options](https://github.com/flux-framework/flux-core/discussions/5784). At LLNL, we implement mpibind through a job shell option, meaning its algorithm can be used completely in user space -- not reliant on the root privileges of a prolog or epilog.

A good example is the `--signal` option, or `-o signal`, which sends a POSIX signal to a job at TIME before it terminates. Our checkpointing users use this a lot.

In [13]:
!flux run -t10s --signal=SIGTERM@5s sleep inf

4.103s: flux-shell[0]: signal: job will expire in 5.0s, sending SIGTERM to job
flux-job: task(s) Terminated


Even our `--out` and `--err` flags from before were job shell options in disguise!

In [14]:
!flux run -o output.stdout.path=/tmp/foo.bar -o output.stderr.path=/tmp/bar.foo hostname

0: stdout redirected to /tmp/foo.bar
0: stderr redirected to /tmp/bar.foo


And, because this is a Flux tutorial, we will again pedantically use dependencies to show the output file from the previous job.

In [15]:
!flux run --dependency=afterok:$(flux job last) cat /tmp/*foo*

auk108.llnl.gov


#### **As a group**

We hope you've gotten some ideas from this chapter on how to apply Flux's submission tools to your workflows. Feel free to use this time to do that! If you want some ideas for things to try, here are some textbook-type problems:

1. Turn this code snippet into a Flux one-liner.
   ```shell
   for job in $(seq 1 100); do
       flux submit -n 1 echo $job
   done
   ```


#### **Other neat command-line tools**: `flux archive`, `flux exec`, `flux pstree`, `flux top`